# Retrieval-Augmented Generation on Historical UAP Records
---

**Students**

* Sofia Di Lucia (Badge No. 2149752)
* Giovanni Andrea Maida (Badge No. 2159404)

**Master's Program:** M.Sc. in Physics of Data

**Academic Year:** 2025–2026

---

**Sections**:
- [Domain and Dataset](#Domain-and-Dataset)
- [Importing Libraries](#Importing-Libraries)
- [Inspecting Dataset](#Inspecting-Dataset)
- [Embedding](#Embedding)
- [Retrieval](#Retrieval)
- [Generation](#Generation)
- [Evaluation](#Evaluation)
- [Acknowledgements](#Acknowledgements)

### Domain and Dataset

The project is based on the *MTS, Department of War UAP Release 1 — Structured Corpus (2026)* dataset (source material at [war.gov/UFO/](https://www.war.gov/UFO/)).

<img src="https://media.mts-in.com/release_1/38-143685-box-incident-summaries-101-172/p137_f1_sketch.webp"
         alt="UFO"
         width="630"
         height="180">

# Importing Libraries

In [1]:
from datasets import load_dataset, get_dataset_config_names, Dataset
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import pandas as pd
from pathlib import Path
import re
import torch.nn.functional as F
from torch import Tensor
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig

# Inspecting Dataset

In [2]:
ufo_dataset = "MTSLIVE/war-gov-uap-release-1"
get_dataset_config_names(ufo_dataset)   # dataset files

['documents', 'pages', 'figures', 'videos']

We will use the `pages` file.

In [3]:
pages = load_dataset(ufo_dataset, "pages", split="train")
pages

Dataset({
    features: ['document_id', 'page_no', 'text', 'has_figures'],
    num_rows: 4239
})

In [4]:
# example of what we are going to use
print(pages[0]['text'])

HEADQUARTERS
AIR MATERIEL COMMAND
WRIGHT FIELD, DAYTON, OHIO

DEC 1 9 1947

SUBJECT: Flying Discs

TO: Chief of Staff
United States Air Force
Washington 25, D. C.
ATTENTION: Director, Research & Development
Major General L. C. Craigie

1. Confirming the recent conversation of the undersigned with Major General L. C. Craigie, 9 December 1947, attached as listed below are copies of the reports from this Headquarters concerning Flying Discs.

2. Comments of Headquarters, Air Force on these letters have never been received by this Command. Continued and recent reports from qualified observers concerning this phenomenon still makes this matter one of concern to Headquarters, Air Materiel Command. Intelligence Department of this Command is continuing the collection and analysis of all available reports.

FOR THE COMMANDING GENERAL:

H. M. McCOY
Colonel, USAF
Chief of Intelligence

2 Attach:
cc ltr to CG, AAF, dtd 23 Sept 47 subj "AMC Opinion Concerning "Flying Discs""
cc ltr to CG, AAF, dtd 

## Check some documents by id

In [5]:
# all documents, each of these is composed of 1 or more pages
IDS = set(pages["document_id"])

Let's see if some pages have less than 20 characters.

In [6]:
null_id = 0     # count for document_id (even if it's just one page)
null_pgs = 0    # count for pages (can share the same id)
doc_ids = []    # to filter later(?)

for page in pages:
    if len(page["text"]) <= 20:
        null_pgs+=1
        if page["document_id"] not in doc_ids:
            null_id+=1
            doc_ids.append(page["document_id"])
        #print(f"doc_id: {page["document_id"]}\n text: {page["text"]}", "\n")

print(f"total null ids (at least one page): {null_id}")
print(f"total null pages: {null_pgs}")
print(f"problematic ids: {doc_ids}")

total null ids (at least one page): 61
total null pages: 225
problematic ids: ['342-hs1-416511228-319-1-flying-discs-1949', '65-hs1-101634279-100-de-26505', '65-hs1-834228961-62-hq-83894-section-1', '65-hs1-834228961-62-hq-83894-section-10', '65-hs1-834228961-62-hq-83894-section-2', '65-hs1-834228961-62-hq-83894-section-3', '65-hs1-834228961-62-hq-83894-section-4', '65-hs1-834228961-62-hq-83894-section-5', '65-hs1-834228961-62-hq-83894-section-6', '65-hs1-834228961-62-hq-83894-section-7', '65-hs1-834228961-62-hq-83894-section-8', '65-hs1-834228961-62-hq-83894-section-9', '65-hs1-834228961-62-hq-83894-serial-130', '65-hs1-834228961-62-hq-83894-serial-164', '65-hs1-834228961-62-hq-83894-serial-438', 'dow-uap-d10-mission-report-middle-east-may-2022', 'dow-uap-d12-mission-report-iraq-may-2022', 'dow-uap-d25-mission-report-greece-january-2024', 'dow-uap-d27-mission-report-united-arab-emirates-october-2023', 'dow-uap-d28-mission-report-iraq-september-2024', 'dow-uap-d3-mission-report-arabian

So there are 61 `document_id` with _at least_ one page with less than 20 characters. If we talk in terms of pages, there are 225 pages almost empty.

In [7]:
count = pages.to_pandas().groupby("document_id").count()
single = list(count[count["page_no"]==1].index)  # Documents of 1 page only
# Print the single paged documents
# for doc in pages.filter(lambda x: x["document_id"] in single): print(f"DOCUMENT: {doc["document_id"].upper()}\n\n{doc["text"]}\n\n\n\n")

In [8]:
# Discard single paged documents containing no information
pattern = re.compile("fbi-photo|nasa-uap-vm|fbi-september-2023-sighting-composite-sketch")  # compile pattern to match discarded documents
useIDS = set(ID for ID in IDS if not pattern.match(ID))  # discard matches

In [9]:
filterPages = pages.filter(lambda x: x["document_id"] in useIDS and x["text"] != '')
# filterPages = pages.filter(lambda x: len(x["text"].strip()) <= 20)

# Embedding

The embedder is choosen from [here](https://huggingface.co/spaces/mteb/leaderboard)

**[Qwen3](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) Model Architecture**: <br>
is designed using dual-encoder and cross-encoder architectures the Embedding model processes a single text segment as input, extracting the semantic representation by utilizing the hidden state vector corresponding to the final [EOS] token. ([ref](https://qwen.ai/blog?id=qwen3-embedding))

<div style="display:flex; gap:20px;">
    <img src="https://miro.medium.com/v2/resize:fit:750/format:webp/1*jzZ_e5Bmvx84zPEa-LhqDQ.png"
         alt="Qwen3 architecture"
         width="500">
    <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*AWPQxx4xpiQGCp5XNNfyYA.png"
         alt="BERT vs Qwen3"
         width="700"
         height="300">
</div>

[article](https://arxiv.org/pdf/2506.05176)
For text embeddings, we utilize LLMs with causal attention, appending an
[EOS] token at the end of the input sequence. The final embedding is derived from the hidden state of the last layer corresponding to this [EOS] token.
To ensure embeddings follow instructions during downstream tasks, we concatenate the instruction and the query into a single input context, while leaving the document unchanged before processing with LLMs. The input format for queries is as follows:
`{Instruction}{Query}<|endoftext|>`

[Harrier-oss-v1](https://huggingface.co/microsoft/harrier-oss-v1-0.6b) is a Decoder-Only architecture as well, and works similarly to Qwen. Harrier is specifically designed to embed text, hence it should achieve similar results even with less parameters, as the leaderboard suggests.

In [10]:
harrier_embedder = "microsoft/harrier-oss-v1-0.6b"
qwen_embedder = "Qwen/Qwen3-Embedding-4B"

To fit the models and the data in the GPU memory, we use the [bitsandbytes](https://huggingface.co/docs/bitsandbytes/index) library, specifically designed to reduce weights' bit representation to $4$ bits, without performance degradation.

In [11]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

In [12]:
# defining the embedding model and the relative tokenizer
# for the tokenizer we need the left padding

# harrier
tokenizer = AutoTokenizer.from_pretrained(harrier_embedder, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModel.from_pretrained(harrier_embedder, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir='models_cache')

# qwen
# tokenizer = AutoTokenizer.from_pretrained(qwen_embedder, padding_side='left', cache_dir='tokenizers_cache')
# model = AutoModel.from_pretrained(qwen_embedder, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

We use [LangChain](https://docs.langchain.com/) to split the pages into chunks of $512$ tokens with $15\%$ overlapping. Using the provided tokenizer, LangChain tries to split the text until the chunks are small enough while trying to keep paragraphs, then sentences, and finally words together where possible.

In [13]:
# function to create the chunked text (input of embedder)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=512, chunk_overlap=77)

pages_chunked = []

for page in filterPages:
    page_text = page['text']
    chunk_list = text_splitter.split_text(page_text)
    for i, chunk in enumerate(chunk_list):
        pages_chunked.append({
            'document_id': page['document_id'],
            'page_no': page['page_no'],
            'chunk_id': i,
            'chunk_text': chunk
    })

chunkedPages = Dataset.from_list(pages_chunked)

In [14]:
# function to extract the last token (EOS) that is a compressed representation of the whole input (instruction+query)
def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

# we "merge" in one string instruction and query
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery: {query}'

def get_embeddings(text_list, max_length=512):

  # Tokenize the input texts
  batch_dict = tokenizer(
    text_list,
    padding='longest',
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
  )
  batch_dict.to(model.device)

  with torch.no_grad(): outputs = model(**batch_dict)
  embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask']).cpu()

  torch.cuda.empty_cache()

  return embeddings

In [15]:
# docsEmbed = torch.load("harrier06DocsEmbed.pt", weights_only=False)  # colab
docsEmbed = torch.load("/mnt/eph/embedding/harrier06DocsEmbed.pt", weights_only=False)  # cv   #### kill eventually

# docsEmbed = torch.load("/mnt/eph/embedding/qwen4DocsEmbed.pt", weights_only=False)

In [16]:
embedPages = chunkedPages.add_column("embeddings", docsEmbed.to(torch.float32).cpu().numpy().tolist())

In [17]:
check = []
keep = []
for i, page in enumerate(embedPages):
    if page["embeddings"] in check: continue
    else:
        keep.append(i)
        check.append(page["embeddings"])

In [18]:
embedPages = embedPages.select(keep)
docsEmbed = docsEmbed[keep]

### Queries

Here we embed only the queries. We need to add the documents part
```python
# input_texts = chunks[:160]  # batchsize 80 for qwen3-0.6 40 for qwen3-4b  ## done in .py script
```

In [19]:
# Each query must come with a one-sentence instruction that describes the task
task = 'Given a document search query, retrieve relevant passages that answer the query'

raw_queries = [
    "What are Jesus's powers?",
    'What was spotted in the sky for the first time?',
    'Have UFOs ever been close to humans (astronauts)?',
    "What patterns emerge across the reported UAP sightings regarding location, altitude, behavior, speed, and time period?",
    "What are the most extravagant sightings?",
    "What is the Saucer's secret?",
    "What are they hiding from us?"
]

queries = [get_detailed_instruct(task, query) for query in raw_queries]

In [20]:
queriesEmbed = get_embeddings(queries)

# Retrieval

Even though the embeddings fit in our machine memory, we decided to use [Faiss](https://github.com/facebookresearch/faiss) to calculate cosine similarity. Faiss is a library for efficient similarity search and clustering of dense vectors, optimized both in terms of computing speed and memory usage.

In [21]:
# retrieving the top-k documents for each query
numpydocs = docsEmbed.to(torch.float32).cpu().numpy()
numpyqueries = queriesEmbed.to(torch.float32).cpu().numpy()
index = faiss.index_factory(docsEmbed.shape[1], "Flat", faiss.METRIC_INNER_PRODUCT)   # cosine similarity
faiss.normalize_L2(numpydocs)
index.add(numpydocs)
faiss.normalize_L2(numpyqueries)

k=5
k_best_distance, k_best_index = index.search(numpyqueries, k)

To avoid excessive text dumping, we show here the selected documents for the first $2$ queries only. The rest is shown in the appendix at the end of the notebook.

In [24]:
## Harrier
for i, query in enumerate(raw_queries[:2]):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}; idx: {idx}\n{embedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.5771710872650146; idx: 3266
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space

In [23]:
## Qwen
for i, query in enumerate(raw_queries[:2]):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}; idx: {idx}\n{embedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.623723030090332; idx: 3272
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space)

For the first query about the powers of Jesus both the models retrieve the same passages, save for minor differences in the tokenization of the text. The scores also seem consistent enough, as the documents are not directly related to the topic, but may contain depositions and "evidence" from common people.

The second query instead may be an example of a too generic question that cannot be answered by these documents. Both the models retrieve passages about flying objects with references to time in general. We would argue that Harrier does a better job, as its documents mostly contain a precise date, while Qwen retrieve passages referring to "hours" time, which is quite not the desired outcome without the reference date. A higher amount of documents may benefit this second query, but it is not given that any good information is retrieved.

# Generation

Suggested parameters from qwen3 model:

> For non-thinking mode, we suggest using Temperature=0.7, TopP=0.8, TopK=20, and MinP=0. For more detailed guidance, please refer to the Best Practices section.

> For thinking mode, use Temperature=0.6, TopP=0.95, TopK=20, and MinP=0 (the default setting in generation_config.json). DO NOT use greedy decoding, as it can lead to performance degradation and endless repetitions. For more detailed guidance, please refer to the Best Practices section.

In [38]:
type(0.7)

float

In [42]:
def generation(input_tokenizer, input_model, messages, max_tokens, temperature=0.7, top_p=0.8, think=False):
  tokenizer = input_tokenizer
  model = input_model

  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=think
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

  # conduct text completion
  generated_ids = model.generate(
      **model_inputs,
      max_new_tokens=max_tokens,  # should be 32768?
      temperature=temperature,
      top_p=top_p
  )
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

  # parsing thinking content
  try:
      # rindex finding 151668 (</think>)
      index = len(output_ids) - output_ids[::-1].index(151668)
  except ValueError:
      index = 0

  thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
  content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

  return thinking_content, content


In [25]:
qwen_generator = "Qwen/Qwen3-4B"

In [23]:
# qwen
tokenizer = AutoTokenizer.from_pretrained(qwen_generator, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModelForCausalLM.from_pretrained(qwen_generator, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [44]:
# FROM DOCS https://huggingface.co/Qwen/Qwen3-4B more or less
system_prompt = "We are impartial detectives investigating the latest FBI UAP encounters. Use the provided documents to answer the Query as best as you can."
think = False
for i, query in enumerate(raw_queries):
    ## through embedPages[int(idx)] you can access any metainformation of the document. It is not necessary possibly even useless to feed it to the generator
    retrDocs = [embedPages[int(idx)]["chunk_text"] for idx in k_best_index[i]]
    prompt = "Provided documents:\n"
    for j, doc in enumerate(retrDocs): prompt += f"Document {str(j + 1)}:\n{doc}.\n\n"    # here we can print metadata (doc_id and page_no) instead of whole text
    prompt += f"\nQuery: {query}"
    if i==0: print(prompt)
    else: print("Query:", query)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    thinking_content, content = generation(tokenizer, model, messages, 16384, think=think)

    print("\nthinking content:", thinking_content)
    print("\ncontent:", content, "\n\n\n")

Provided documents:
Document 1:
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space) with power and great glory" (

/mnt/eph/miniconda3/envs/nlp/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



thinking content: 

content: Based on the provided documents, the text suggests that Jesus possesses several "powers" which are interpreted through a metaphorical or symbolic lens, often relating to supernatural or extraterrestrial themes. Here are the key points about Jesus's powers as described in the documents:

1. **Levitation**: The text mentions that Jesus had the power to levitate, which is described as a "power of levitation" that is also attributed to "men from outer space."

2. **Passing Through Doors**: Jesus is said to have the ability to pass through doors, which is interpreted as a supernatural or extraterrestrial power.

3. **Walking on Water**: Jesus is described as having the power to walk on water, which is a well-known biblical miracle but here is framed within the context of extraterrestrial abilities.

4. **Healing the Sick**: Jesus is said to have the power to heal the sick, a common miracle attributed to him in religious texts.

5. **Ascension in a "Cloud"**: Th

# Evaluation

Generators can be found [here](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard#/)

In [11]:
test_pages = filterPages.shuffle(seed=32).select(range(15))

In [12]:
# da definire
test_generator = "Qwen/Qwen2.5-Coder-14B-Instruct"
#test_generator = "Qwen/Qwen3-4B-Instruct-2507"

In [ ]:
# da definire
test_tokenizer = AutoTokenizer.from_pretrained(test_generator, padding_side='left', cache_dir='tokenizers_cache')
test_model = AutoModelForCausalLM.from_pretrained(test_generator, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

In [ ]:
# two quastions/answers for each passage
system_prompt = """You are creating a benchmark for evaluating a RAG system.
Given a passage, generate exactly two question/answer pair.
Output format:

Question 1: ...
Gold Answer 1: ...
Question 2: ...
Gold Anser 2: ...
"""
qa_pairs = []
for num, passage in enumerate(test_pages):
    page_text = passage["text"]
    prompt = f"Passage:\n{page_text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    thinking_content, content = generation(test_tokenizer, test_model, messages, 16384)    # max tokens to be decided
    qa_pairs.append(content)

    print(f"Passage {str(num + 1)}:", thinking_content)
    print("content:", content, "\n-------------------------")

In [ ]:
questions1 = re.findall(r'Question 1:\s*(.+?)(?=\n|$)', '\n'.join(qa_pairs))
gold_answers1 = re.findall(r'Gold Answer 1:\s*(.+)', '\n'.join(qa_pairs))
questions2 = re.findall(r'Question 2:\s*(.+?)(?=\n|$)', '\n'.join(qa_pairs))
gold_answers2 = re.findall(r'Gold Answer 2:\s*(.+)', '\n'.join(qa_pairs))

In [ ]:
def save_file(new_file_name, obj_to_save):
  with open(new_file_name, 'w') as f:
    for line in obj_to_save:
        f.write(f"{line}\n")
  return print(f"File {new_file_name} saved")

def load_file(file_name):
    with open(file_name, "r") as f:
        obj_saved = [line.strip() for line in f]
    return obj_saved

In [ ]:
save_file("questions1.txt", questions1)

In [ ]:
save_file("questions2.txt", questions2)

In [ ]:
save_file("gold_answers1.txt", gold_answers1)

In [ ]:
save_file("gold_answers2.txt", gold_answers2)

Here we use our RAG to generate answer to the given test-questions.

In [ ]:
# FROM DOCS https://huggingface.co/Qwen/Qwen3-4B more or less
system_prompt = """Given a question and the relative passage, generate the most accurate answer.
Output format:

Question 1: ...
Answer 1: ...
Question 2: ...
Answer 2: ...
"""
answers = []
for num, passage in enumerate(test_pages):
  prompt = f"Question 1: {questions1[num]}\nQuestion 2: {questions2[num]}\nPassage: {passage['text']}"
  messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": prompt}
  ]
  thinking_content, content = generation(tokenizer, model, messages, 16384)    # max tokens to be decided

  answers.append(content)

  print(f"Passage {str(num + 1)}:", thinking_content)
  print("content:", content, "\n-------------------------")

In [ ]:
rag_answers1 = re.findall(r'Answer 1:\s*(.+)', '\n'.join(answers))
rag_answers2 = re.findall(r'Answer 2:\s*(.+)', '\n'.join(answers))

In [ ]:
save_file("rag_answers1.txt", rag_answers1)

In [ ]:
save_file("rag_answers2.txt", rag_answers2)

Now that we have the questions, gold answers and answers we can use them as input for our test_generator that will act as a judge providing a score.

In [ ]:
judge_input = {'quest': {'quest_1': questions1,
                         'quest_2': questions2},
               'gold_ans': {'quest_1': gold_answers1,
                            'quest_2': gold_answers2},
               'rag_ans': {'quest_1': rag_answers1,
                            'quest_2': rag_answers2}
               }

In [127]:
# to try
system_prompt = """You are a judge. Given a question, a gold answer and another second answer: you will give me a score for the second answer.
The score is a number between 0 and 100. The higher the score, the better the answer. The gold answer has a 100 score.
Output format for each passage X:
Passage X:
Question 1: ...
RAG answer score: .../100
Question 2: ...
RAG answer score: .../100
"""
scores = []
for num in range(len(test_pages)):
    
    quest1 = judge_input['quest']['quest_1'][num]
    quest2 = judge_input['quest']['quest_2'][num]
    goldansw1 = judge_input['gold_ans']['quest_1'][num]
    goldansw2 = judge_input['gold_ans']['quest_2'][num]
    ragansw1 = judge_input['rag_ans']['quest_1'][num]
    ragansw2 = judge_input['rag_ans']['quest_2'][num]

    prompt = f"Question 1: {quest1}\nGold answer 1: {goldansw1}\nRAG answer 1: {ragansw1}\n\nQuestion 2: {quest2}\nGold answer 2: {goldansw2}\nRAG answer 2: {ragansw2}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    thinking_content, content = generation(test_tokenizer, test_model, messages, 16384)

    scores.append(content)

    print(f"Passage {str(num + 1)}:", thinking_content)
    print("content:", content, "\n-------------------------")

Passage 1: 
content: Passage 1:
Question 1: Who is Richard F. Shaver mentioned in the memorandum?
RAG answer score: 85/100

Question 2: What was the purpose of the memorandum?
RAG answer score: 80/100 
-------------------------
Passage 2: 
content: Passage 1:
Question 1: Who moderated the colloquium on UFOs held at Pocantico in 1997?
RAG answer score: 100/100

Question 2: What was the main focus of the colloquium organized by Laurance Rockefeller?
RAG answer score: 100/100 
-------------------------
Passage 3: 
content: Passage 1:
Question 1: What was the purpose of Mr. Wacks' letter to Mr. Joseph F. Perry?
RAG answer score: 95/100

Question 2: To whom should Mr. Perry direct further inquiries about the return of the photographs?
RAG answer score: 98/100 
-------------------------
Passage 4: 
content: Passage 1:
Question 1: Who is the commanding general addressed in the passage?
RAG answer score: 100/100

Question 2: What is the subject of the forwarded information?
RAG answer score: 7

In [ ]:
scores[:2]

In [ ]:
for passage in scores:
    scores1 = re.findall(r'Question (\d+):.*?RAG answer score: (\d+)/100', scores, re.DOTALL)

In [ ]:
scores1

# Acknowledgements

Put here solutions that have been inspired by similar projects publicly available on the web.

- Dataset: https://huggingface.co/datasets/MTSlive/war-gov-uap-release-1

# Appendix

## Appendix A - Other Queries output

In [25]:
## Harrier
for i, query in enumerate(raw_queries):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}; idx: {idx}\n{embedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.5771710872650146; idx: 3266
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space

In [24]:
## Qwen
for i, query in enumerate(raw_queries):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}; idx: {idx}\n{embedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.623723030090332; idx: 3272
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space)